In [1]:
# Imports

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

In [2]:
# Load dataset

csv_path = "sepsis_icu_synthetic.csv"

df = pd.read_csv(csv_path)

df.head()

,subject_id,age,gender,weight_kg,height_cm,bmi,ethnicity,insurance,hr_mean,hr_max,...,apache_iv,qsofa,sirs_criteria,gcs_total,icu_los_hours,hospital_admit_source,icu_admit_time_hour,day_of_week,readmission_30day,sepsis_label
0,37464,72,M,99.5,160.6,38.6,Black,Medicare,83.0,98.7,...,47.9,2,1,12,27.0,OR,15,2,1,1
1,34024,62,M,101.2,164.5,37.4,Other,Private,97.5,105.8,...,35.0,1,1,6,21.5,Transfer,21,2,0,0
2,65291,74,M,64.0,167.2,22.9,White,Private,100.6,105.8,...,29.0,1,1,11,46.7,ED,19,6,0,0
3,79118,87,M,75.1,180.5,23.0,White,Medicare,95.5,114.7,...,35.6,1,1,7,1.7,ED,8,2,0,0
4,34897,61,M,52.5,158.4,NaN,Hispanic,Private,86.7,106.7,...,34.6,1,1,8,64.0,Transfer,9,2,0,0


In [3]:
# Clean data

df["gender"] = df["gender"].replace({"Mael": "M"})

categorical_cols = df.select_dtypes(include="object").columns
numeric_cols = df.select_dtypes(exclude="object").columns

df[categorical_cols] = df[categorical_cols].fillna("Unknown")
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

df.head()

,subject_id,age,weight_kg,height_cm,bmi,hr_mean,hr_max,hr_min,hr_std,sbp_mean,...,ethnicity_Black,ethnicity_Hispanic,ethnicity_Other,ethnicity_White,insurance_Medicare,insurance_Private,insurance_Self-pay,hospital_admit_source_ED,hospital_admit_source_OR,hospital_admit_source_Transfer
0,37464,72,99.5,160.6,38.6,83.0,98.7,73.7,7.4,113.8,...,True,False,False,False,True,False,False,False,True,False
1,34024,62,101.2,164.5,37.4,97.5,105.8,87.9,4.0,110.9,...,False,False,True,False,False,True,False,False,False,True
2,65291,74,64.0,167.2,22.9,100.6,105.8,91.4,7.2,119.5,...,False,False,False,True,False,True,False,True,False,False
3,79118,87,75.1,180.5,23.0,95.5,114.7,85.1,3.6,132.5,...,False,False,False,True,True,False,False,True,False,False
4,34897,61,52.5,158.4,25.6,86.7,106.7,73.2,11.5,102.0,...,False,True,False,False,False,True,False,False,False,True


In [4]:
label = "sepsis_label"

X = df.drop(columns=[label])
y = df["sepsis_label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (5000, 83)
y shape: (5000,)


In [5]:
# Data disparity

sepsis_counts = df["sepsis_label"].value_counts().reset_index()
sepsis_counts.columns = ["sepsis_label", "count"]

sepsis_counts

,sepsis_label,count
0,0,4250
1,1,750


In [6]:
# Split dataset into training & testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (4000, 83)
X_test: (1000, 83)
y_train: (4000,)
y_test: (1000,)


# Logistic Regression

In [7]:
# Create model & hyperparameters

from sklearn.preprocessing import StandardScaler, RobustScaler
from models.logistic_regression import LRCV

model_standardscaler=StandardScaler()
Cs = [0.01, 0.1, 1, 10, 100]

LR_standardscaler = LRCV(Cs=Cs, scaler=model_standardscaler)

In [8]:
LR_standardscaler.fit(X_train, y_train)

In [9]:
y_pred = LR_standardscaler.predict(X_test)

y_prob = LR_standardscaler.predict_proba(X_test)[:, 1]

In [13]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("Best Cs:",LR_standardscaler.best_Cs())

Confusion Matrix:
[[850   0]
 [  0 150]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       850
           1       1.00      1.00      1.00       150

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
ROC-AUC: 1.0
Best Cs: [0.1]


# Random Forest